In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load and clean
df = pd.read_csv("construction_estimates.csv")
target = "Total_Estimate"

df = df.drop_duplicates()
df = df.dropna(subset=[target]).reset_index(drop=True)

# Exclude target from input features
X = df.drop(columns=[target], errors="ignore")
y = df[target]

X = X.select_dtypes(exclude=["complex"])
X = X.dropna(axis=1, how="all")

# 2. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

# 3. Identify feature types
numeric_cols = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_cols = [
    col for col in X_train.columns
    if col not in numeric_cols
]

# 4. Preprocessing
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

# 5. Build model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# 6. Train and evaluate
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 4))
print("R2:", round(r2_score(y_test, pred), 4))

# 7. Save
os.makedirs("saved_models", exist_ok=True)

joblib.dump(model, "saved_models/construction_estimate_model.pkl")
joblib.dump({
    "target": target,
    "features": X.columns.tolist(),
    "numeric_features": numeric_cols,
    "categorical_features": categorical_cols
}, "saved_models/construction_estimate_model_metadata.pkl")

print("Construction estimate model and metadata saved.")

MAE: 2383.8717
RMSE: 3019.2814
R2: 0.9684
Construction estimate model and metadata saved.
